# OligoTox v0.3 — Data Gap Analysis & Prioritization

In [ ]:
import pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

df = pd.read_csv(os.path.expanduser('~/oligotox_datasets/processed/curated_oligo_tox_master_v3.csv'))
ep_cols = [c for c in df.columns if c.startswith('endpoint_')]
summary = pd.DataFrame({'endpoint': [c.replace('endpoint_','') for c in ep_cols],
                        'n_pos': [df[c].sum() for c in ep_cols]})
summary['n_total'] = len(df)
summary['pos_rate'] = summary['n_pos'] / summary['n_total']
summary['gap_to_12'] = (12 - summary['n_pos']).clip(lower=0)
summary['status'] = pd.cut(summary['n_pos'], bins=[-1,1,3,7,999],
                            labels=['SPARSE','THIN','MODERATE','OK'])
print(summary.to_string(index=False))


In [ ]:
colors = {'SPARSE':'#d62728','THIN':'#ff7f0e','MODERATE':'#2ca02c','OK':'#1f77b4'}
ss = summary.sort_values('n_pos')
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
bc = [colors.get(str(s), '#aaa') for s in ss['status']]
axes[0].barh(ss['endpoint'], ss['n_pos'], color=bc)
axes[0].axvline(4, color='red', ls='--', lw=1, label='Min trainable (4)')
axes[0].axvline(12, color='blue', ls='--', lw=1, label='Target (12)')
axes[0].set_xlabel('Positive Training Examples'); axes[0].set_title('v0.3 Coverage')
axes[0].legend()
axes[1].barh(ss['endpoint'], ss['gap_to_12'], color='#d62728', alpha=0.7)
axes[1].set_xlabel('Additional Positives Needed'); axes[1].set_title('Gap to 12-positive Target')
plt.tight_layout()
plt.savefig(os.path.expanduser('~/oligotox_datasets/docs/endpoint_gap_analysis.png'), dpi=150, bbox_inches='tight')
print('Chart saved.')


## Dataset Priority Queue

| Priority | Dataset | Est. new positives | Endpoints unlocked |
|---|---|---|---|
| 1 | Papargyri 2020 768-gapmer caspase Table S1 | +50-100 | hepatocellular_injury ✓, offtarget ✓ |
| 2 | Inotersen/eplontersen NDA platelet tables | +8-15 | thrombocytopenia ✓, coagulation ✓ |
| 3 | Patisiran cyno complement NDA Table 9 | +6-10 | complement ✓, cytokine ✓ |
| 4 | Nusinersen/tofersen CSF NfL histopath | +5-8 | cns_neurotoxicity ✓ |
| 5 | Any DART+ high-LNA gapmer EFD data | +2-5 | reproductive_dev_tox ✓ |
| 6 | NIH OligoTox Challenge Phase 2 (Dec 2026) | TBD | All endpoints |